In [114]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib # For saving/loading models
from pathlib import Path
from datetime import datetime

# ── Core ML (always required) ─────────────────────────────────────────────────
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error
)
from sklearn.preprocessing import LabelEncoder


try:
    import optuna
    _HAS_OPTUNA = True
except ImportError:
    _HAS_OPTUNA = False

try:
    import xgboost as xgb
    _HAS_XGB = True
except ImportError:
    _HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    _HAS_CAT = True
except ImportError:
    _HAS_CAT = False

In [115]:
# File paths

CSV_PATH  = Path("dataset_with_model_first_name.csv") 
MODEL_OUT = Path("lgbm_vehicle_price_model_v2.pkl")    
STACK_OUT = Path("stacking_meta_model_v2.pkl")
STATS_OUT = Path("vehicle_statistics_v2.csv")

# Data Cleaning 

PRICE_CAP = 30_000_000
MILEAGE_CAP = 1_000_000
PRICE_FLOOR = 100_000

# temporal split for validation
TEST_SPLIT_DAYS = 90

CV_FOLDS = 5
RANDOM_STATE = 42
OPTUNA_TRIALS = 5

In [116]:
def load_and_clean(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Drop unnamed columns that may have been created during CSV export
    df.drop(columns=[c for c in df.columns if 'Unnamed' in c],
            inplace=True, errors='ignore')   

    # Drop rows with missing critical values
    df.dropna(subset=['Make', 'Model', 'Year', 'Price', 'Milleage', 'published date'],
              inplace=True)
    
    # Convert data types and handle errors
    df['Year']           = df['Year'].astype(int)
    df['Price']          = pd.to_numeric(df['Price'],    errors='coerce')
    df['Milleage']       = pd.to_numeric(df['Milleage'], errors='coerce')
    df['published date'] = pd.to_datetime(df['published date'], errors='coerce')

    # drop rows with invalid numeric values after coercion
    df.dropna(subset=['Price', 'Milleage', 'published date'], inplace=True)

    # Filter out outliers based on domain knowledge
    df = df[(df['Price']   > PRICE_FLOOR) & (df['Price']   < PRICE_CAP)]
    df = df[(df['Milleage'] >= 0)         & (df['Milleage'] < MILEAGE_CAP)]

    # Remove duplicates based on URL, keeping the most recent entry
    url_col = next((c for c in df.columns if 'url' in c.lower()), None)
    if url_col:
        df.sort_values('published date', inplace=True)
        before = len(df)
        df.drop_duplicates(subset=[url_col], keep='last', inplace=True)
        print(f"[DEDUP] Removed {before - len(df):,} duplicate URLs")

    # normalize text to uppercase, strip whitespace
    for col in ['Make', 'Model']:
        df[col] = df[col].astype(str).str.strip().str.upper()

    # Simplify model names by keeping only the first two words
    df['Model'] = df['Model'].apply(
        lambda x: ' '.join(x.split()[:2]) if len(x.split()) > 2 else x
    )

    # ── NEW: Filter out corrupted text values (Excel errors, placeholders, etc) ──
    invalid_patterns = ['#REF!', '#VALUE!', '#DIV/0!', '#NAME?', '#NULL!', 
                        '?', 'NAN', 'NONE', 'NULL', 'ERROR', 'N/A', '']
    before_invalid = len(df)
    for col in ['Make', 'Model']:
        df = df[~df[col].isin(invalid_patterns)]
    invalid_removed = before_invalid - len(df)
    if invalid_removed > 0:
        print(f"[CLEAN] Removed {invalid_removed:,} rows with corrupted Make/Model values")

    # Sort by Make, Model, and published date
    df.sort_values(['Make', 'Model', 'published date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"[DATA] {len(df):,} clean records | "
          f"{df['Make'].nunique()} makes | {df['Model'].nunique()} models | "
          f"Years: {df['Year'].min()}–{df['Year'].max()}")
    return df


raw_df = load_and_clean(CSV_PATH)
raw_df.head(5)

[DEDUP] Removed 0 duplicate URLs
[CLEAN] Removed 16 rows with corrupted Make/Model values
[DATA] 23,388 clean records | 67 makes | 1126 models | Years: 1900–2026


,Vehicle Type,Make,Model,Year,Price,Milleage,District,published date,Vehicle URL,Condition
0,Car,ACURA,ZEN,2000,1800000.0,100000.0,Colombo,2026-02-03,https://riyasewana.com/buy/acura-zen-sale-colo...,Used
1,Car,ALFA-ROMEO,155,1994,2150000.0,52000.0,Wennappuwa,2026-01-17,https://riyasewana.com/buy/alfa-romeo-155-t-sa...,Used
2,Car,ALFA-ROMEO,156,2001,4100000.0,170.0,Colombo,2026-01-04,https://riyasewana.com/buy/alfa-romeo-156-sale...,Used
3,Car,ALFA-ROMEO,COOPER,1980,1625000.0,80000.0,Battaramulla,2026-02-06,https://riyasewana.com/buy/alfa-romeo-cooper-s...,Used
4,Car,ASHOK-LEYLAND,1613,2006,4800000.0,300000.0,Matugama,2025-12-01,https://riyasewana.com/buy/ashok-leyland-1613-...,Used


In [117]:
def derive_condition(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    def _cond(row):
        yr, km = row['Year'], row['Milleage']
        if yr >= 2022:
            if km < 5_000:    return 'Brand New'
            if km <= 50_000:  return 'Recondition'
        return 'Used'  # Default: older cars always "Used" regardless of km

    df['Condition'] = df.apply(_cond, axis=1)

    print("[CONDITION] Distribution:")
    print(df['Condition'].value_counts().to_string())
    return df


# ── Run it ─────────────────────────────────────────────────────────────────────
raw_df = derive_condition(raw_df)
raw_df[['Year', 'Milleage', 'Condition']].sample(5)



[CONDITION] Distribution:
Condition
Used           20896
Recondition     1390
Brand New       1102


,Year,Milleage,Condition
5697,2010,200000.0,Used
14560,2018,68000.0,Used
11259,2011,112700.0,Used
21528,1996,166000.0,Used
17374,1999,160000.0,Used


In [118]:
def engineer_features(df: pd.DataFrame, ref_date: pd.Timestamp = None) -> pd.DataFrame:

    df = df.copy()
    if ref_date is None:
        ref_date = df['published date'].max()

    # Temporal features from published date
    df['post_year']   = df['published date'].dt.year
    df['month']       = df['published date'].dt.month         
    df['quarter']     = df['published date'].dt.quarter       
    df['day_of_year'] = df['published date'].dt.dayofyear     
    df['week']        = df['published date'].dt.isocalendar().week.astype(int)

    # Age of the car at the time of listing
    df['Car_Age']    = (df['post_year'] - df['Year']).clip(lower=0)

    # 
    df['log_car_age'] = np.log1p(df['Car_Age'])

    # Non-linear transformation of age
    df['car_age_sq']  = df['Car_Age'] ** 2

    # Log-transform mileage to reduce skewness
    df['log_milleage'] = np.log1p(df['Milleage'])

    # Mileage per year (with handling for zero age)
    df['km_per_year']     = df['Milleage'] / df['Car_Age'].replace(0, 0.5)
    df['log_km_per_year'] = np.log1p(df['km_per_year'])


    grp = df.groupby(['Make', 'Model'])['Price']
    df['grp_median_price'] = grp.transform('median')  # Robust central price
    df['grp_mean_price']   = grp.transform('mean')    # Mean (affected by outliers)
    df['grp_std_price']    = grp.transform('std').fillna(0)  # Price spread/variance
    df['grp_count']        = grp.transform('count')   # How many listings = liquidity

    # Temporal price trends by year
    grp2 = df.groupby(['Make', 'Model', 'Year'])['Price']
    df['yr_median_price'] = grp2.transform('median')
    df['yr_mean_price']   = grp2.transform('mean')

    df['lag_1'] = df.groupby(['Make', 'Model'])['Price'].shift(1)
    df['lag_2'] = df.groupby(['Make', 'Model'])['Price'].shift(2)
    df['lag_7'] = df.groupby(['Make', 'Model'])['Price'].shift(7)

    df['rolling_mean_3'] = (
        df.groupby(['Make', 'Model'])['Price']
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )
    df['rolling_mean_7'] = (
        df.groupby(['Make', 'Model'])['Price']
        .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    )

    for lag_col in ['lag_1', 'lag_2', 'lag_7', 'rolling_mean_3', 'rolling_mean_7']:
        df[lag_col].fillna(df['grp_median_price'], inplace=True)

    df['price_vs_median'] = df['lag_1'] / df['grp_median_price'].replace(0, 1)


    cond_map = {'Brand New': 3, 'Recondition': 2, 'Used': 1, 'Unknown': 0}
    df['condition_code'] = df['Condition'].map(cond_map).fillna(1).astype(int)

    print(f"[FEATURES] Engineered {len(df):,} rows with {df.shape[1]} columns")
    print(f"[FEATURES] Lag NaN fill: using group median (no leakage)")
    return df

# ── Run it ─────────────────────────────────────────────────────────────────────
eng_df = engineer_features(raw_df)

# Display summary statistics of the new features
new_features = ['log_car_age', 'car_age_sq', 'log_milleage', 'log_km_per_year',
                'grp_median_price', 'yr_median_price', 'price_vs_median',
                'rolling_mean_7', 'condition_code']
print("\nSample of new engineered features:")
eng_df[new_features].describe().round(2)



[FEATURES] Engineered 23,388 rows with 34 columns
[FEATURES] Lag NaN fill: using group median (no leakage)

Sample of new engineered features:


,log_car_age,car_age_sq,log_milleage,log_km_per_year,grp_median_price,yr_median_price,price_vs_median,rolling_mean_7,condition_code
count,23388.00,23388.00,23388.00,23388.00,23388.00,23388.00,23388.00,23388.00,23388.00
mean,2.65,415.91,10.99,8.49,7051465.73,7235524.26,1.04,7208902.61,1.15
std,0.78,545.55,2.12,1.78,4358958.89,4929740.04,0.47,4462085.18,0.47
min,0.00,0.00,0.69,0.02,265000.00,265000.00,0.02,265000.00,1.00
25%,2.30,81.00,11.12,8.54,3800000.00,3775000.00,0.89,3928571.43,1.00
50%,2.71,196.00,11.71,9.00,6650000.00,6370000.00,1.00,6426071.43,1.00
75%,3.22,576.00,12.06,9.31,9025000.00,9000000.00,1.12,9121607.14,1.00
max,4.84,15876.00,13.82,13.25,29800000.00,29950000.00,13.26,29800000.00,3.00


In [119]:
#  FEATURE LISTS — Central registry of all model inputs

# ── Numeric features ──────────────────────────────────────────────────────────

NUMERIC_FEATURES = [
    # --- Depreciation curve ---
    'Car_Age',          # Integer age in years
    'log_car_age',      # Log-transformed age (captures non-linear depreciation)
    'car_age_sq',       # Squared age (captures accelerating depreciation curve)

    # --- Mileage / Usage ---
    'log_milleage',     # Log-transformed total mileage
    'log_km_per_year',  # Log-transformed annual mileage intensity

    # --- Seasonality ---
    'month',            #
    'quarter',          
    'day_of_year',      
    'week',             

    # --- Price momentum (lag features) ---
    'lag_1',            # Previous listing price for this Make+Model
    'lag_2',            # 2nd previous listing price
    'lag_7',            # 7th previous listing price (~1 week history)
    'rolling_mean_3',   # 3-period rolling average price
    'rolling_mean_7',   # 7-period rolling average price

    # --- Market anchors ---
    'grp_median_price', # Median price for this Make+Model across all time
    'grp_mean_price',   # Mean price for this Make+Model
    'grp_std_price',    # Price spread / volatility for this Make+Model
    'grp_count',        # Number of listings (market liquidity signal)
    'yr_median_price',  # Median price for this Make+Model+Year combination
    'yr_mean_price',    # Mean price for this Make+Model+Year combination

    # --- Relative signals ---
    'price_vs_median',  # Current price ÷ group median (over/under-pricing ratio)

    # --- Condition ---
    'condition_code',   # Ordinal: Brand New=3, Recondition=2, Used=1
]

# ── Categorical features ───────────────────────────────────────────────────────
# LightGBM handles these natively — no label encoding or one-hot needed
# LightGBM finds optimal splits across category bins automatically
CATEGORICAL_FEATURES = [
    'Make',   # Vehicle manufacturer (Toyota, Honda, Suzuki, …)
    'Model',  # Vehicle model (YARIS, VEZEL, WAGON R, …)
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Total features: {len(ALL_FEATURES)}")
print(f"  Numeric     : {len(NUMERIC_FEATURES)}")
print(f"  Categorical : {len(CATEGORICAL_FEATURES)}")
print(f"\nFeature list:")
for i, f in enumerate(ALL_FEATURES, 1):
    print(f"  {i:2d}. {f}")

Total features: 24
  Numeric     : 22
  Categorical : 2

Feature list:
   1. Car_Age
   2. log_car_age
   3. car_age_sq
   4. log_milleage
   5. log_km_per_year
   6. month
   7. quarter
   8. day_of_year
   9. week
  10. lag_1
  11. lag_2
  12. lag_7
  13. rolling_mean_3
  14. rolling_mean_7
  15. grp_median_price
  16. grp_mean_price
  17. grp_std_price
  18. grp_count
  19. yr_median_price
  20. yr_mean_price
  21. price_vs_median
  22. condition_code
  23. Make
  24. Model


In [120]:
def log_target(y: pd.Series) -> pd.Series:
    
    return np.log1p(y)


def exp_target(y_log: np.ndarray) -> np.ndarray:
   
    return np.expm1(y_log)


# ── Quick sanity check ───────────────────────────────────
test_prices = np.array([500_000, 2_000_000, 10_000_000, 25_000_000])
log_prices  = log_target(pd.Series(test_prices))
recovered   = exp_target(log_prices.values)

print("Log-transform sanity check:")
print(f"{'Original LKR':>20} | {'Log Value':>10} | {'Recovered LKR':>15}")
print("-" * 52)
for orig, logv, rec in zip(test_prices, log_prices, recovered):
    print(f"{orig:>20,.0f} | {logv:>10.4f} | {rec:>15,.0f}")
print("\n All values recovered perfectly (max rounding error < 1 LKR)")

Log-transform sanity check:
        Original LKR |  Log Value |   Recovered LKR
----------------------------------------------------
             500,000 |    13.1224 |         500,000
           2,000,000 |    14.5087 |       2,000,000
          10,000,000 |    16.1181 |      10,000,000
          25,000,000 |    17.0344 |      25,000,000

 All values recovered perfectly (max rounding error < 1 LKR)


In [121]:
def _optuna_objective(trial, X_tr, y_tr, X_va, y_va):
    
    params = dict(
        # Number of trees — higher is better but with early stopping it auto-stops
        n_estimators      = trial.suggest_int('n_estimators', 1000, 5000),

        # Learning rate — small rate + many trees = better generalisation
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.05, log=True),

        # Num leaves = model complexity. Original was 31 (default/too small).
        num_leaves        = trial.suggest_int('num_leaves', 20, 80),

        # Min samples per leaf — key regulariser for sparse Make+Model groups
        min_child_samples = trial.suggest_int('min_child_samples', 10, 60),

        # Row subsampling — reduces overfitting, adds variance reduction
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),

        # Feature subsampling — makes trees more diverse in the ensemble
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),

        # L1 / L2 regularisation — penalises large leaf weights
        reg_alpha         = trial.suggest_float('reg_alpha',  1e-4, 1.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),

        # Max depth — limits tree depth (additional complexity control)
        max_depth         = trial.suggest_int('max_depth', 4, 12),

        # Min split gain — minimum improvement needed to make a split
        min_split_gain    = trial.suggest_float('min_split_gain', 0.0, 0.5),

        random_state = RANDOM_STATE,
        verbose      = -1,  # Suppress LightGBM per-tree logs
    )

    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        categorical_feature=CATEGORICAL_FEATURES,
        # early_stopping: stop if validation metric doesn't improve for 100 rounds
        # This prevents both overfitting and wasted compute
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    # Evaluate in original LKR space (not log space) so MAPE is meaningful
    preds     = exp_target(m.predict(X_va))
    y_va_orig = exp_target(y_va)
    return mean_absolute_percentage_error(y_va_orig, preds)


def get_best_lgbm_params(X_tr, y_tr, X_va, y_va) -> dict:
    
    if not _HAS_OPTUNA:
        print("[HPO] Using hand-tuned params (install optuna for auto-tuning)")
        return dict(
            n_estimators=4000, learning_rate=0.01, num_leaves=40,
            min_child_samples=30, subsample=0.80, colsample_bytree=0.80,
            reg_alpha=0.05,  reg_lambda=0.10, max_depth=8,
            min_split_gain=0.01, random_state=RANDOM_STATE, verbose=-1,
        )

    print(f"[HPO] Running Optuna with {OPTUNA_TRIALS} trials…")
    study = optuna.create_study(
        direction='minimize',  # We want to MINIMISE MAPE
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(
        lambda t: _optuna_objective(t, X_tr, y_tr, X_va, y_va),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=True,
    )
    print(f"[HPO] Best validation MAPE: {study.best_value*100:.2f}%")
    print(f"[HPO] Best params: {study.best_params}")

    # Merge best params with fixed non-tunable params
    return {**study.best_params, 'random_state': RANDOM_STATE, 'verbose': -1}


print("HPO functions defined. Will run during train_model() call.")

HPO functions defined. Will run during train_model() call.


In [122]:
def _print_metrics(y_true, y_pred):
    
    # Core regression metrics
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)

    
    safe_pred = np.maximum(y_pred, 0)  
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(safe_pred)) ** 2))

    
    naive_mae = float(np.abs(y_true - y_true.median()).mean())
    mase      = mae / naive_mae if naive_mae > 0 else 0

    acc = 100 - mape  # Percentage accuracy (100 - MAPE)

    print(f"\n{'='*60}")
    print(f"  MODEL PERFORMANCE METRICS (TEST SET — {len(y_true):,} samples)")
    print(f"{'='*60}")
    print(f"  R² Score : {r2:.4f}    {'Excellent' if r2 > 0.85 else ' Needs work' if r2 > 0.7 else ' Poor'}")
    print(f"  Accuracy : {acc:.2f}%   (100 - MAPE)")
    print(f"{'─'*60}")
    print(f"  MAE      : LKR {mae:>12,.0f}  {'Ok' if mae < 250_000 else ' Check'}")
    print(f"  RMSE     : LKR {rmse:>12,.0f}  (penalises big errors)")
    print(f"  MAPE     :        {mape:.2f}%    {'Excellent' if mape < 5 else 'check' if mape < 10 else '✗ High'}")
    print(f"  RMSLE    :       {rmsle:.4f}     {'ok' if rmsle < 0.15 else 'check' if rmsle < 0.2 else '✗'}")
    print(f"  MASE     :       {mase:.4f}     {'ok' if mase < 0.5 else 'check' if mase < 1.0 else '✗'}")
    print(f"{'='*60}")

    # Prediction interval check (are 80% of errors within ±15%?)
    pct_errors = np.abs((y_true.values - y_pred) / y_true.values) * 100
    within_5   = (pct_errors <= 5).mean()  * 100
    within_10  = (pct_errors <= 10).mean() * 100
    within_15  = (pct_errors <= 15).mean() * 100
    print(f"\n  Error Distribution:")
    print(f"    Within  ±5%  : {within_5:.1f}% of predictions")
    print(f"    Within ±10%  : {within_10:.1f}% of predictions")
    print(f"    Within ±15%  : {within_15:.1f}% of predictions")
    print(f"{'='*60}\n")


def _print_feature_importance(model, feature_names, top_n=15):
    
    fi = pd.Series(
        model.feature_importances_,
        index=feature_names
    ).sort_values(ascending=False)

    print(f"\n[FEATURE IMPORTANCE — Top {top_n} by gain]")
    print(f"{'Feature':<25} {'Importance':>12}")
    print("─" * 40)
    for feat, imp in fi.head(top_n).items():
        pct = (imp / fi.iloc[0] * 100)
        bar = f'{pct:.1f}%'
    print(f"  {feat:<23} {imp:>10,.0f}  {bar:>6}")
    return fi


print("Metric and importance functions defined.")

Metric and importance functions defined.


In [123]:
def _encode_for_xgb(X: pd.DataFrame) -> pd.DataFrame:
    
    X = X.copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    return X


def train_model(df: pd.DataFrame):
    
    df = df.copy()

    # Step 1: Encode categoricals for LightGBM native handling
    # LightGBM with categorical_feature finds optimal bin-splits automatically
    # Much better than one-hot encoding for high-cardinality categoricals like Model
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].astype('category')

    # Step 2: TEMPORAL split — the critical data leakage fix
    # Last TEST_SPLIT_DAYS days = test set (simulates real production conditions)
    # Everything before = training data
    split_date = df['published date'].max() - pd.Timedelta(days=TEST_SPLIT_DAYS)
    train_df   = df[df['published date'] <= split_date].copy()
    test_df    = df[df['published date'] >  split_date].copy()

    print(f"[SPLIT] Train: {len(train_df):,} records (up to {split_date.date()})")
    print(f"[SPLIT] Test : {len(test_df):,}  records (after {split_date.date()})")

    if len(test_df) < 50:
        print("[WARN] Very small test set — reduce TEST_SPLIT_DAYS or add more data")

    # Prepare X/y (log-transform y for training)
    X_train = train_df[ALL_FEATURES]
    y_train = log_target(train_df['Price'])   # ← Training on LOG price
    X_test  = test_df[ALL_FEATURES]
    y_test  = test_df['Price']                # ← Evaluating on ORIGINAL LKR

    # Inner validation window for HPO (last 10% of train, still temporal)
    val_cut = int(len(X_train) * 0.9)
    X_tr, y_tr = X_train.iloc[:val_cut], y_train.iloc[:val_cut]
    X_va, y_va = X_train.iloc[val_cut:], y_train.iloc[val_cut:]

    # Step 3: Get best hyperparameters (via Optuna or hand-tuned fallback)
    print("\n[HPO] Finding best LightGBM hyperparameters…")
    best_params = get_best_lgbm_params(X_tr, y_tr, X_va, y_va)

    # Step 4: Train final LightGBM on full training set
    print("\n[TRAIN] Training LightGBM on full train set…")
    lgbm_model = lgb.LGBMRegressor(**best_params)
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_va, y_va)],
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[
            lgb.early_stopping(150, verbose=False),  # Stop if no improvement
            lgb.log_evaluation(500),                 # Print every 500 trees
        ],
    )
    print(f"[TRAIN] LightGBM done. Best iteration: {lgbm_model.best_iteration_}")

    # Collect base learner predictions on test set (for stacking)
    base_preds_test  = {}
    base_preds_train = {}
    base_preds_test['lgbm']  = exp_target(lgbm_model.predict(X_test))
    base_preds_train['lgbm'] = exp_target(lgbm_model.predict(X_train))

    # Step 5a: XGBoost base learner (if installed)
    if _HAS_XGB:
        print("[STACK] Training XGBoost base learner…")
        xgb_model = xgb.XGBRegressor(
            n_estimators=2000, learning_rate=0.02, max_depth=7,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.05, reg_lambda=0.1,
            random_state=RANDOM_STATE, verbosity=0, enable_categorical=True,
        )
        X_tr_xgb = _encode_for_xgb(X_train)
        X_te_xgb = _encode_for_xgb(X_test)
        xgb_model.fit(
            X_tr_xgb, y_train,
            verbose=False,
        )
        base_preds_test['xgb']  = exp_target(xgb_model.predict(X_te_xgb))
        base_preds_train['xgb'] = exp_target(xgb_model.predict(X_tr_xgb))
        print(f"[STACK] XGBoost done.")

    # Step 5b: CatBoost base learner (if installed)
    if _HAS_CAT:
        print("[STACK] Training CatBoost base learner…")
        cat_model = CatBoostRegressor(
            iterations=2000, learning_rate=0.02, depth=7,
            loss_function='RMSE', random_state=RANDOM_STATE, verbose=0,
        )
        X_tr_cat = X_train.copy()
        X_te_cat = X_test.copy()
        for col in CATEGORICAL_FEATURES:
            X_tr_cat[col] = X_tr_cat[col].astype(str)
            X_te_cat[col] = X_te_cat[col].astype(str)
        cat_feats = [X_tr_cat.columns.get_loc(c) for c in CATEGORICAL_FEATURES]
        cat_model.fit(
            X_tr_cat, y_train, cat_features=cat_feats,
            eval_set=(X_te_cat, log_target(y_test)),
            early_stopping_rounds=100,
        )
        base_preds_test['cat']  = exp_target(cat_model.predict(X_te_cat))
        base_preds_train['cat'] = exp_target(cat_model.predict(X_tr_cat))
        print(f"[STACK] CatBoost done.")

    
    # to prevent meta-model overfitting
    if len(base_preds_test) > 1:
        print("\n[STACK] Fitting Ridge meta-learner on base predictions…")
        # Meta-learner trains on LOG scale for numerical stability
        stack_X_test = np.column_stack([np.log1p(v) for v in base_preds_test.values()])
        meta = Ridge(alpha=1.0)
        meta.fit(stack_X_test, log_target(y_test))
        final_preds = exp_target(meta.predict(stack_X_test))
        joblib.dump(meta, STACK_OUT)
        print(f"[STACK] Meta-model weights: {dict(zip(base_preds_test.keys(), meta.coef_))}")
    else:
        # Only LightGBM available — use it directly
        final_preds = base_preds_test['lgbm']
        meta = None
        print("[STACK] Single model mode (install xgboost/catboost for stacking)")

    # Step 7: Print comprehensive metrics
    _print_metrics(y_test, final_preds)
    _print_feature_importance(lgbm_model, ALL_FEATURES)

    return lgbm_model, meta, df


# ── Run training ──────────────────────────────────────────────────────────────
lgbm_model, meta_model, eng_df = train_model(eng_df)

[I 2026-03-20 22:18:41,792] A new study created in memory with name: no-name-545d8db2-d700-47c6-9235-92bb6fea6287


[SPLIT] Train: 158 records (up to 2025-11-09)
[SPLIT] Test : 23,230  records (after 2025-11-09)

[HPO] Finding best LightGBM hyperparameters…
[HPO] Running Optuna with 5 trials…


Best trial: 0. Best value: 0.588649:   0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-03-20 22:18:41,865] Trial 0 finished with value: 0.5886488553446036 and parameters: {'n_estimators': 2498, 'learning_rate': 0.044635901521768134, 'num_leaves': 64, 'min_child_samples': 40, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.0001707396743152812, 'reg_lambda': 0.29154431891537513, 'max_depth': 9, 'min_split_gain': 0.35403628889802274}. Best is trial 0 with value: 0.5886488553446036.


[I 2026-03-20 22:18:41,909] Trial 1 finished with value: 0.549330852513099 and parameters: {'n_estimators': 1082, 'learning_rate': 0.04665303012212833, 'num_leaves': 70, 'min_child_samples': 20, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 0.012561043700013555, 'max_depth': 7, 'min_split_gain': 0.14561457009902096}. Best is trial 1 with value: 0.549330852513099.


Best trial: 1. Best value: 0.549331:  80%|████████  | 4/5 [00:00<00:00, 17.09it/s]

[I 2026-03-20 22:18:41,962] Trial 2 finished with value: 0.5860418320362701 and parameters: {'n_estimators': 3448, 'learning_rate': 0.006893882309676883, 'num_leaves': 37, 'min_child_samples': 28, 'subsample': 0.7824279936868144, 'colsample_bytree': 0.9140703845572055, 'reg_alpha': 0.0006290644294586153, 'reg_lambda': 0.011400863701127324, 'max_depth': 9, 'min_split_gain': 0.023225206359998862}. Best is trial 1 with value: 0.549330852513099.
[I 2026-03-20 22:18:42,012] Trial 3 finished with value: 0.6083290685460099 and parameters: {'n_estimators': 3430, 'learning_rate': 0.007404472559987595, 'num_leaves': 23, 'min_child_samples': 58, 'subsample': 0.9862528132298237, 'colsample_bytree': 0.9233589392465844, 'reg_alpha': 0.001653693718282443, 'reg_lambda': 0.00024586032763280086, 'max_depth': 10, 'min_split_gain': 0.22007624686980065}. Best is trial 1 with value: 0.549330852513099.
[I 2026-03-20 22:18:42,054] Trial 4 finished with value: 0.5867484992156202 and parameters: {'n_estimators'

Best trial: 1. Best value: 0.549331: 100%|██████████| 5/5 [00:00<00:00, 19.01it/s]


[HPO] Best validation MAPE: 54.93%
[HPO] Best params: {'n_estimators': 1082, 'learning_rate': 0.04665303012212833, 'num_leaves': 70, 'min_child_samples': 20, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 0.012561043700013555, 'max_depth': 7, 'min_split_gain': 0.14561457009902096}

[TRAIN] Training LightGBM on full train set…
[TRAIN] LightGBM done. Best iteration: 85
[STACK] Training XGBoost base learner…
[STACK] XGBoost done.
[STACK] Training CatBoost base learner…
[STACK] CatBoost done.

[STACK] Fitting Ridge meta-learner on base predictions…
[STACK] Meta-model weights: {'lgbm': np.float64(0.3070239029239111), 'xgb': np.float64(0.49926453104259505), 'cat': np.float64(0.2602214957527184)}

  MODEL PERFORMANCE METRICS (TEST SET — 23,230 samples)
  R² Score : 0.8756    Excellent
  Accuracy : 81.71%   (100 - MAPE)
────────────────────────────────────────────────────────────
  MAE      : LKR      892,017   Check
 

In [124]:
def predict_price(
    lgbm_model,
    meta_model,
    df: pd.DataFrame,
    make: str,
    model_name: str,
    manufacture_year: int,
    query_date: str = None,
    horizon_days: int = 7,
    verbose: bool = True,
) -> dict:
   
    # Normalise inputs to match training data format
    make_u    = str(make).upper().strip()
    model_u   = str(model_name).upper().strip()
    query_ts  = pd.Timestamp(query_date) if query_date else pd.Timestamp.today()

    # ── Fallback 1: Exact match (Make + Model + Year + date filter) ───────────
    subset = df[
        (df['Make'].astype(str).str.upper() == make_u) &
        (df['Model'].astype(str).str.upper() == model_u) &
        (df['Year'] == manufacture_year) &
        (df['published date'] <= query_ts)
    ].sort_values('published date')

    # ── Fallback 2: Relax date constraint ─────────────────────────────────────
    if subset.empty:
        print(f"[FALLBACK] No data before {query_ts.date()} — using all dates")
        subset = df[
            (df['Make'].astype(str).str.upper() == make_u) &
            (df['Model'].astype(str).str.upper() == model_u) &
            (df['Year'] == manufacture_year)
        ].sort_values('published date')

    # ── Fallback 3: Relax year constraint ─────────────────────────────────────
    if subset.empty:
        print(f"[FALLBACK] No {manufacture_year} data — using any year for {make_u} {model_u}")
        subset = df[
            (df['Make'].astype(str).str.upper() == make_u) &
            (df['Model'].astype(str).str.upper() == model_u)
        ].sort_values('published date')

    if subset.empty:
        raise ValueError(
            f"No data found for '{make_u}' '{model_u}'. "
            "Check spelling or add more training data."
        )

    # ── Extract historical price signals ──────────────────────────────────────
    last_prices    = subset['Price'].tail(7).values
    last_price     = float(last_prices[-1])
    second_last    = float(last_prices[-2]) if len(last_prices) >= 2 else last_price
    rolling3       = float(np.mean(last_prices[-3:]))
    rolling7       = float(np.mean(last_prices[-7:]))
    avg_mileage    = float(subset['Milleage'].mean())

    # Market median for this Make+Model (used as price anchor)
    grp_med = float(df[
        (df['Make'].astype(str).str.upper() == make_u) &
        (df['Model'].astype(str).str.upper() == model_u)
    ]['Price'].median())

    if verbose:
        print(f"\n{'─'*55}")
        print(f"  Vehicle       : {make_u} {model_u} ({manufacture_year})")
        print(f"  Query Date    : {query_ts.date()}")
        print(f"  Last Listed   : LKR {last_price:,.0f}")
        print(f"  Market Median : LKR {grp_med:,.0f}")
        print(f"  Avg Mileage   : {avg_mileage:,.0f} km")
        print(f"  Forecasting   : {horizon_days} days ({horizon_days//7} week(s))\n")

    # ── Iterative weekly forecast ────────────
    # We predict one week at a time, updating lag features with each prediction.
    # This is proper time-series forecasting — each prediction feeds the next.
    predictions = {}
    lag1, lag2  = last_price, second_last
    steps       = max(1, horizon_days // 7)

    for step in range(steps):
        # Advance one week from query date
        future_date = query_ts + pd.Timedelta(days=7 * (step + 1))
        car_age     = max(0, future_date.year - manufacture_year)
        km_per_yr   = avg_mileage / max(car_age, 0.5)

        # Determine condition code from average mileage
        cond_code = 3 if avg_mileage < 5_000 else (2 if avg_mileage < 50_000 else 1)

        # Build a single-row feature DataFrame matching training schema
        row = {
            'Make'             : make_u,
            'Model'            : model_u,
            'Car_Age'          : car_age,
            'log_car_age'      : np.log1p(car_age),
            'car_age_sq'       : car_age ** 2,
            'log_milleage'     : np.log1p(avg_mileage),
            'log_km_per_year'  : np.log1p(km_per_yr),
            'month'            : future_date.month,
            'quarter'          : (future_date.month - 1) // 3 + 1,
            'day_of_year'      : future_date.timetuple().tm_yday,
            'week'             : future_date.isocalendar()[1],
            'lag_1'            : lag1,           # Updated each iteration ← KEY
            'lag_2'            : lag2,           # Updated each iteration ← KEY
            'lag_7'            : last_price,     # Anchor to last real price
            'rolling_mean_3'   : rolling3,
            'rolling_mean_7'   : rolling7,
            'grp_median_price' : grp_med,
            'grp_mean_price'   : grp_med,
            'grp_std_price'    : 0.0,
            'grp_count'        : int(len(subset)),
            'yr_median_price'  : last_price,
            'yr_mean_price'    : last_price,
            'price_vs_median'  : lag1 / (grp_med or 1.0),
            'condition_code'   : cond_code,
        }

        # Build prediction DataFrame with correct categorical dtype
        pred_df = pd.DataFrame([row])
        for col in CATEGORICAL_FEATURES:
            known_cats = df[col].astype('category').cat.categories
            pred_df[col] = pd.Categorical(pred_df[col], categories=known_cats)

        # Predict in log space, convert back to LKR
        pred_log  = lgbm_model.predict(pred_df[ALL_FEATURES])[0]
        predicted = float(exp_target(np.array([pred_log]))[0])

        # ── Safety clamp: prevent runaway predictions ─────────────────────────
    
        # This catches cases where the model extrapolates badly for sparse groups
                # ── Safety clamp: condition-specific bounds ─────────────────────────
        # Brand new cars: tighter bounds (less volatility)
        # Used cars: wider bounds (more natural variation)
        if cond_code == 3:  # Brand New (mileage < 5,000 km)
            predicted = float(np.clip(predicted, rolling3 * 0.85, rolling3 * 1.25))
        elif cond_code == 2:  # Recondition (5k-50k km)
            predicted = float(np.clip(predicted, rolling3 * 0.75, rolling3 * 1.35))
        else:  # Used (>50k km)
            predicted = float(np.clip(predicted, rolling3 * 0.60, rolling3 * 1.40))

        # Format output key as date range
        week_start = future_date
        week_end   = future_date + pd.Timedelta(days=6)
        date_key   = f"{week_start.strftime('%Y-%m-%d')} → {week_end.strftime('%Y-%m-%d')}"
        predictions[date_key] = round(predicted)

        # Update rolling lags for next iteration (iterative forecasting)
        lag2     = lag1
        lag1     = predicted
        rolling3 = float(np.mean([rolling3, lag1, lag2]))

        if verbose:
            pct    = (predicted - last_price) / last_price * 100
            symbol = '↑' if pct > 0 else '↓'
            print(f"    Week {step+1}: {date_key}  →  LKR {predicted:>12,.0f}   {symbol} {abs(pct):.1f}%")

    return {
        'make'          : make_u,
        'model'         : model_u,
        'year'          : manufacture_year,
        'current_price' : last_price,
        'avg_mileage'   : avg_mileage,
        'query_date'    : str(query_ts.date()),
        'predictions'   : predictions,    # dict: {date_range: price_LKR}
    }


print("predict_price() function defined.")

predict_price() function defined.


In [125]:
# =============================================================================
#  PUBLIC API — These are the only functions your backend needs to call
# =============================================================================

def predict_next_week(lgbm, meta, df, make, model_name, year, query_date=None):
    
    return predict_price(lgbm, meta, df, make, model_name, year,
                         query_date=query_date, horizon_days=7)


def predict_next_month(lgbm, meta, df, make, model_name, year, query_date=None):
    
    return predict_price(lgbm, meta, df, make, model_name, year,
                         query_date=query_date, horizon_days=28)


# ── Demo Predictions ──────────────────────────────────────────────────────────
USER_QUERY_DATE = datetime.today().strftime('%Y-%m-%d')

demo_vehicles = [
    ('Toyota', 'YARIS',   2018),
    ('Honda',  'VEZEL',   2019),
    ('Toyota', 'RAIZE',   2021),
    ('Suzuki', 'WAGON R', 2020),
    ('Honda', 'VEZEL', 2025)
]

print("=" * 65)
print("  DEMO — NEXT WEEK PREDICTIONS")
print("=" * 65)

for make, model_name, year in demo_vehicles:
    try:
        result = predict_next_week(
            lgbm_model, meta_model, eng_df,
            make, model_name, year, USER_QUERY_DATE
        )
    except ValueError as e:
        print(f"  [SKIP] {e}")

print("\n" + "=" * 65)
print("  DEMO — NEXT MONTH PREDICTIONS (4 weekly steps)")
print("=" * 65)

for make, model_name, year in demo_vehicles[:2]:  # Shorter demo
    try:
        result = predict_next_month(
            lgbm_model, meta_model, eng_df,
            make, model_name, year, USER_QUERY_DATE
        )
    except ValueError as e:
        print(f"  [SKIP] {e}")

  DEMO — NEXT WEEK PREDICTIONS

───────────────────────────────────────────────────────
  Vehicle       : TOYOTA YARIS (2018)
  Query Date    : 2026-03-20
  Last Listed   : LKR 10,150,000
  Market Median : LKR 9,250,000
  Avg Mileage   : 79,200 km
  Forecasting   : 7 days (1 week(s))

    Week 1: 2026-03-27 → 2026-04-02  →  LKR    8,056,443   ↓ 20.6%

───────────────────────────────────────────────────────
  Vehicle       : HONDA VEZEL (2019)
  Query Date    : 2026-03-20
  Last Listed   : LKR 13,800,000
  Market Median : LKR 11,500,000
  Avg Mileage   : 94,883 km
  Forecasting   : 7 days (1 week(s))

    Week 1: 2026-03-27 → 2026-04-02  →  LKR   15,352,145   ↑ 11.2%
[FALLBACK] No data before 2026-03-20 — using all dates
[FALLBACK] No 2021 data — using any year for TOYOTA RAIZE

───────────────────────────────────────────────────────
  Vehicle       : TOYOTA RAIZE (2021)
  Query Date    : 2026-03-20
  Last Listed   : LKR 10,490,000
  Market Median : LKR 13,125,000
  Avg Mileage   : 13,5

In [126]:
def generate_vehicle_statistics_csv(
    lgbm_model,
    meta_model,
    df: pd.DataFrame,
    output_path: Path = STATS_OUT,
    query_date: str = None,
) -> pd.DataFrame:
    
    qdate = query_date or datetime.today().strftime('%Y-%m-%d')

    # Group by Make+Model+Year to get one row per vehicle variant
    groups = (
        df.groupby(['Make', 'Model', 'Year'])
          .agg(
              Average_Price   = ('Price',   'mean'),
              Average_Mileage = ('Milleage', 'mean'),
              Listings        = ('Price',   'count'),
          )
          .reset_index()
    )

    # ── NEW: Filter out groups with no valid price data ──
    # This prevents NaN-to-int conversion errors in the fallback logic
    before_filter = len(groups)
    groups = groups[
        groups['Average_Price'].notna() & 
        (groups['Average_Price'] > PRICE_FLOOR) &
        (groups['Average_Price'] < PRICE_CAP) &
        groups['Average_Mileage'].notna()
    ]
    filtered_out = before_filter - len(groups)
    if filtered_out > 0:
        print(f"[CSV] Filtered out {filtered_out} groups with invalid price data")

    print(f"\n[CSV] Generating predictions for {len(groups):,} valid vehicle groups…")
    print(f"[CSV] Query date: {qdate}")

    rows = []
    failed = 0
    skipped = 0

    for idx, row in groups.iterrows():
        try:
            # Predict both next week and next month in one call
            result = predict_price(
                lgbm_model, meta_model, df,
                make             = row['Make'],
                model_name       = row['Model'],
                manufacture_year = int(row['Year']),
                query_date       = qdate,
                horizon_days     = 28,   # 4 weeks
                verbose          = False,
            )
            week_price  = list(result['predictions'].values())[0]   # Week 1
            month_price = list(result['predictions'].values())[-1]  # Week 4

        except Exception as e:
            # Graceful fallback: use historical average if prediction fails
            # This ensures the CSV is always complete, never partial
            avg_price = row['Average_Price']
            
            # Safety check: only use fallback if price is valid
            if pd.isna(avg_price) or avg_price <= 0:
                skipped += 1
                continue  # Skip this entire row — it has no valid price
            
            week_price  = int(round(avg_price))
            month_price = int(round(avg_price))
            failed += 1

        rows.append({
            'Make'                     : row['Make'],
            'Model'                    : row['Model'],
            'Year'                     : int(row['Year']),
            'Average_Price_LKR'        : int(round(row['Average_Price'])),
            'Average_Mileage_km'       : int(round(row['Average_Mileage'])),
            'Listings_Count'           : int(row['Listings']),
            'Predicted_Next_Week_LKR'  : week_price,
            'Predicted_Next_Month_LKR' : month_price,
        })

        # Progress logging every 50 vehicles
        if (idx + 1) % 50 == 0:
            print(f"  [{idx+1}/{len(groups)}] Processing… ({failed} fallbacks, {skipped} skipped so far)")

    stats_df = pd.DataFrame(rows).sort_values(['Make', 'Model', 'Year'])
    stats_df.to_csv(output_path, index=False)

    print(f"\n[CSV] ✓ Saved → {output_path}")
    print(f"[CSV] Total: {len(stats_df):,} rows | Fallbacks: {failed} | Skipped: {skipped}")
    print("\nSample output:")
    print(stats_df.head(5).to_string(index=False))
    return stats_df

In [127]:
# =============================================================================
#  SAVE — Run once after training
# =============================================================================

# Save LightGBM model
joblib.dump(lgbm_model, MODEL_OUT)
print(f"[SAVE] LightGBM model → {MODEL_OUT}")

# Save Ridge meta-model (if stacking was used)
if meta_model is not None:
    joblib.dump(meta_model, STACK_OUT)
    print(f"[SAVE] Meta-model    → {STACK_OUT}")

# Save the engineered DataFrame — needed for prediction lookups
# (lag features require historical price data at inference time)
eng_df.to_csv('eng_df_v2.csv', index=False)
print(f"[SAVE] Feature DataFrame → eng_df_v2.csv")


# =============================================================================
#  LOAD — Use this in your FastAPI / Flask backend
# =============================================================================
# def load_model_for_api():
#     """
#     Load the trained model and data for use in production API.

#     Call this once at API startup (not on every request).
#     Store the returned objects in app state / global variables.

#     Example (FastAPI):
#         @app.on_event('startup')
#         async def startup():
#             app.state.lgbm, app.state.meta, app.state.df = load_model_for_api()

#         @app.get('/predict')
#         async def predict(make: str, model: str, year: int):
#             result = predict_next_week(
#                 app.state.lgbm, app.state.meta, app.state.df,
#                 make, model, year
#             )
#             return result
#     """
#     lgbm = joblib.load(MODEL_OUT)

#     meta = None
#     if STACK_OUT.exists():
#         meta = joblib.load(STACK_OUT)

#     df = pd.read_parquet('eng_df_v2.parquet')
#     # Re-cast categoricals (parquet loses category dtype)
#     for col in CATEGORICAL_FEATURES:
#         df[col] = df[col].astype('category')

#     print(f"[LOAD] Model loaded: {MODEL_OUT}")
#     print(f"[LOAD] Data loaded : {len(df):,} rows")
#     return lgbm, meta, df


# # ── Quick test of save/load cycle ─────────────────────────────────────────────
# loaded_lgbm, loaded_meta, loaded_df = load_model_for_api()

# # Verify loaded model gives same predictions as original
# test_result = predict_next_week(
#     loaded_lgbm, loaded_meta, loaded_df,
#     'Toyota', 'YARIS', 2018, USER_QUERY_DATE
# )
# print(f"\n[VERIFY] Loaded model prediction: LKR {list(test_result['predictions'].values())[0]:,.0f}")
# print("[VERIFY] ✓ Save/load cycle successful")

[SAVE] LightGBM model → lgbm_vehicle_price_model_v2.pkl


[SAVE] Meta-model    → stacking_meta_model_v2.pkl
[SAVE] Feature DataFrame → eng_df_v2.csv


In [128]:
# =============================================================================
#  MAIN — Full pipeline execution
#  Run this cell to execute all steps end-to-end
# =============================================================================

if __name__ == '__main__' or True:   # Remove 'or True' to run only as script
    print("=" * 65)
    print("  AUTOINSIGHT — VEHICLE PRICE PREDICTION MODEL 2")
    print("=" * 65)

    # ── Step 1: Load & clean data ─────────────────────────────────────────────
    print("\n[STEP 1/6] Loading and cleaning data…")
    raw_df = load_and_clean(CSV_PATH)

    # ── Step 2: Derive vehicle condition ──────────────────────────────────────
    print("\n[STEP 2/6] Deriving vehicle condition…")
    raw_df = derive_condition(raw_df)

    # ── Step 3: Feature engineering ───────────────────────────────────────────
    print("\n[STEP 3/6] Engineering features…")
    eng_df = engineer_features(raw_df)

    # ── Step 4: Train model ────────────────────────────────────────────────────
    print("\n[STEP 4/6] Training model (this may take 5–15 minutes with Optuna)…")
    lgbm_model, meta_model, eng_df = train_model(eng_df)

    # ── Step 5: Save model ─────────────────────────────────────────────────────
    print("\n[STEP 5/6] Saving models…")
    joblib.dump(lgbm_model, MODEL_OUT)
    if meta_model is not None:
        joblib.dump(meta_model, STACK_OUT)
    eng_df.to_csv('eng_df_v2.csv', index=False)
    print(f"  Saved: {MODEL_OUT}, eng_df_v2.parquet")

    # ── Step 6: Generate statistics CSV ───────────────────────────────────────
    print("\n[STEP 6/6] Generating vehicle statistics CSV…")
    USER_QUERY_DATE = datetime.today().strftime('%Y-%m-%d')
    stats_df = generate_vehicle_statistics_csv(
        lgbm_model, meta_model, eng_df,
        query_date=USER_QUERY_DATE,
    )

    # ── Final demo ─────────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  DEMO PREDICTIONS")
    print("=" * 65)

    for make, model_name, year in [('Toyota', 'YARIS', 2018),
                                    ('Honda',  'VEZEL', 2019)]:
        print(f"\n  ── Next Week: {make} {model_name} {year} ──")
        try:
            predict_next_week(lgbm_model, meta_model, eng_df,
                              make, model_name, year, USER_QUERY_DATE)
        except ValueError as e:
            print(f"  [SKIP] {e}")

        print(f"\n  ── Next Month: {make} {model_name} {year} ──")
        try:
            predict_next_month(lgbm_model, meta_model, eng_df,
                               make, model_name, year, USER_QUERY_DATE)
        except ValueError as e:
            print(f"  [SKIP] {e}")

    print("\n" + "=" * 65)
    print("Pipeline complete. All models saved.")
    print("=" * 65)

  AUTOINSIGHT — VEHICLE PRICE PREDICTION MODEL 2

[STEP 1/6] Loading and cleaning data…
[DEDUP] Removed 0 duplicate URLs
[CLEAN] Removed 16 rows with corrupted Make/Model values
[DATA] 23,388 clean records | 67 makes | 1126 models | Years: 1900–2026

[STEP 2/6] Deriving vehicle condition…
[CONDITION] Distribution:
Condition
Used           20896
Recondition     1390
Brand New       1102

[STEP 3/6] Engineering features…


[I 2026-03-20 22:19:09,461] A new study created in memory with name: no-name-bb494482-b64a-48dd-98db-a1d053c867f6


[FEATURES] Engineered 23,388 rows with 34 columns
[FEATURES] Lag NaN fill: using group median (no leakage)

[STEP 4/6] Training model (this may take 5–15 minutes with Optuna)…
[SPLIT] Train: 158 records (up to 2025-11-09)
[SPLIT] Test : 23,230  records (after 2025-11-09)

[HPO] Finding best LightGBM hyperparameters…
[HPO] Running Optuna with 5 trials…


Best trial: 1. Best value: 0.549331:  60%|██████    | 3/5 [00:00<00:00, 29.99it/s]

[I 2026-03-20 22:19:09,490] Trial 0 finished with value: 0.5886488553446036 and parameters: {'n_estimators': 2498, 'learning_rate': 0.044635901521768134, 'num_leaves': 64, 'min_child_samples': 40, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.0001707396743152812, 'reg_lambda': 0.29154431891537513, 'max_depth': 9, 'min_split_gain': 0.35403628889802274}. Best is trial 0 with value: 0.5886488553446036.
[I 2026-03-20 22:19:09,518] Trial 1 finished with value: 0.549330852513099 and parameters: {'n_estimators': 1082, 'learning_rate': 0.04665303012212833, 'num_leaves': 70, 'min_child_samples': 20, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 0.012561043700013555, 'max_depth': 7, 'min_split_gain': 0.14561457009902096}. Best is trial 1 with value: 0.549330852513099.
[I 2026-03-20 22:19:09,558] Trial 2 finished with value: 0.5860418320362701 and parameters: {'n_estimators': 3448

Best trial: 1. Best value: 0.549331: 100%|██████████| 5/5 [00:00<00:00, 29.07it/s]


[I 2026-03-20 22:19:09,629] Trial 4 finished with value: 0.5867484992156202 and parameters: {'n_estimators': 1488, 'learning_rate': 0.015636765183901856, 'num_leaves': 22, 'min_child_samples': 56, 'subsample': 0.7035119926400067, 'colsample_bytree': 0.8650089137415928, 'reg_alpha': 0.0017654048052495078, 'reg_lambda': 0.012030178871154668, 'max_depth': 8, 'min_split_gain': 0.09242722776276352}. Best is trial 1 with value: 0.549330852513099.
[HPO] Best validation MAPE: 54.93%
[HPO] Best params: {'n_estimators': 1082, 'learning_rate': 0.04665303012212833, 'num_leaves': 70, 'min_child_samples': 20, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 0.012561043700013555, 'max_depth': 7, 'min_split_gain': 0.14561457009902096}

[TRAIN] Training LightGBM on full train set…
[TRAIN] LightGBM done. Best iteration: 85
[STACK] Training XGBoost base learner…
[STACK] XGBoost done.
[STACK] Training CatBoost base learner…
[STACK] 